# 01 · From source records to analytical evidence

Question: which defects are formatting issues, and which require stewardship? All values below are calculated from a newly generated fictional dataset. The data-generating process creates the patterns; no inference about real enterprises is justified.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd  # noqa: E402 — repository source bootstrap above
from intelligent_mdm.ingestion import generate, validate_bundle  # noqa: E402
bundle = generate("demo", seed=42)
assert validate_bundle(bundle).empty
bundle.metadata

{'company': 'Lumen Distribution',
 'size': 'demo',
 'seed': 42,
 'as_of': '2026-09-30',
 'currency': 'EUR',
 'period_start': '2024-10-01',
 'period_end': '2026-09-30',
 'generator_version': 1,
 'synthetic': True}

In [2]:
pd.DataFrame({"table": bundle.tables.keys(), "rows": [len(t) for t in bundle.tables.values()]})

,table,rows
0,customers,120
1,customer_sources,252
2,products,48
3,product_sources,101
4,suppliers,12
5,supplier_sources,26
6,categories,4
7,orders,1400
8,order_lines,4201
9,sales,1321


## Inspect distributions and missingness before transforming

Sample variance uses ddof=1. Missingness is measured before cleaning; a malformed email is present but invalid. Correlations and univariate flags are descriptive.

In [3]:
from intelligent_mdm.eda import profile, missing_patterns, missing_by_source, outliers
raw = bundle.tables["customer_sources"]
lines = bundle.tables["order_lines"]
profile(lines)["numeric"]

,field,count,mean,median,mode,std,variance,min,p05,q1,q3,p95,max
0,quantity,4201,5.003095,4.00,3.00,9.064911,82.172609,1.00,1.00,3.00,6.00,10.00,225.00
1,unit_price,4201,61.881133,60.29,21.47,36.803155,1354.472226,4.19,14.20,27.95,89.70,130.06,148.86
2,unit_cost,4201,41.426075,41.49,66.83,23.384605,546.839732,3.09,9.37,17.58,62.64,75.07,77.80
3,discount,4201,0.014866,0.00,0.00,0.027599,0.000762,0.00,0.00,0.00,0.00,0.05,0.10
4,net_amount,4201,302.622466,204.30,77.40,677.791585,459401.432320,3.99,28.10,92.20,384.84,745.28,27586.00


In [4]:
missing_patterns(raw)

,missing_fields,rows,pct
0,Complete row,200,79.365079
1,email,17,6.746032
2,region,14,5.555556
3,tax_id,12,4.761905
4,"email, region",3,1.190476
5,"tax_id, email",3,1.190476
6,"tax_id, region",2,0.793651
7,"tax_id, email, region",1,0.396825


In [5]:
missing_by_source(raw)

,name,tax_id,email,phone,region,created_at,updated_at,record_id
source,,,,,,,,
administration,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
sales,0.0,15.0,20.0,0.0,16.666667,0.0,0.0,0.0


In [6]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3), layout="constrained")
axes[0].hist(lines.quantity, bins=35, color="#7267EF")
axes[0].set(title="All order quantities", xlabel="Quantity", ylabel="Lines")
axes[1].boxplot(lines.quantity, vert=False)
axes[1].set(title="Bulk purchases remain visible", xlabel="Quantity")
display(fig)
plt.close(fig)

Matplotlib is building the font cache; this may take a moment.


/var/folders/lz/1hrp8smd0dvbhf6rqhw7w0vr0000gn/T/ipykernel_62955/3061816411.py:7: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  axes[1].boxplot(lines.quantity, vert=False)


<Figure size 1000x300 with 2 Axes>

## Compare transformations without concealing original data

Normalization is reversible as a snapshot. It fixes formatting and quarantines malformed emails, but does not guess tax keys or regions. A lower completeness score can be the correct result.

In [7]:
from intelligent_mdm.cleaning import normalize
from intelligent_mdm.data_quality import assess
clean, audit = normalize(raw)
comparison = pd.DataFrame({"original": assess(raw)["dimensions"], "normalized": assess(clean)["dimensions"]})
comparison

,original,normalized
completeness,97.266314,97.045855
uniqueness,51.282051,51.282051
validity,73.958333,100.000000
consistency,76.068376,76.068376
referential_integrity,NaN,NaN
freshness,95.238095,95.238095


In [8]:
audit.head(12)

,record,field,before,after,operation,reason
0,CR00013-1,name,SOLSTICE WORKS 0013,Solstice Works 0013,normalize,Canonical format; invalid email becomes missing
1,CR00034-1,name,VELA SUPPLY 0034,Vela Supply 0034,normalize,Canonical format; invalid email becomes missing
2,CR00000-1,name,ASTR STUDIO 0000,Astr Studio 0000,normalize,Canonical format; invalid email becomes missing
3,CR00024-1,name,VLA TRADING 0024,Vla Trading 0024,normalize,Canonical format; invalid email becomes missing
4,CR00050-1,name,ASTER STUDIO 0050,Aster Studio 0050,normalize,Canonical format; invalid email becomes missing
5,CR00114-1,name,VELA WORKS 0114,Vela Works 0114,normalize,Canonical format; invalid email becomes missing
6,CR00040-1,name,ASTR LABS 0040,Astr Labs 0040,normalize,Canonical format; invalid email becomes missing
7,CR00014-1,name,VELA WORKS 0014,Vela Works 0014,normalize,Canonical format; invalid email becomes missing
8,CR00070-1,name,ASTER TRADING 0070,Aster Trading 0070,normalize,Canonical format; invalid email becomes missing
9,CR00036-1,name,COBALT WORKS 0036,Cobalt Works 0036,normalize,Canonical format; invalid email becomes missing


## Compare statistical flags with constructed legitimate extremes

Bulk-purchase labels are evaluation-only metadata. They are not error labels, and they never enter these algorithms. IQR can flag many ordinary heavy-tail values; z-scores depend strongly on this distribution.

In [9]:
flags = outliers(lines.quantity)
pd.DataFrame({"method": ["IQR", "|z|>3"], "flagged_lines": [int(flags.iqr.sum()), int(flags.z_flag.sum())]})

,method,flagged_lines
0,IQR,99
1,|z|>3,20


In [10]:
assert len(raw) == len(clean)
assert audit.shape[0] > 0
assert raw.email.isna().sum() < clean.email.isna().sum()
print("Executed: original rows preserved, invalid emails quarantined and logged.")

Executed: original rows preserved, invalid emails quarantined and logged.


## Interpretation

The generated data support source-owner review of missing/contradictory identifiers. Normalization improves format validity but cannot resolve missingness or identity conflicts. See the printed values and `docs/METHODOLOGY.md` for denominators, rules and limitations.